# Fixed income valuation and loss-quantile risk

This notebook is an educational walk-through of a coupon bond and of value-at-risk on a **simulated** return path. It is not trading advice and it does not use market datasets.

This repository is an analytical and educational quantitative-finance portfolio. It does not represent trading advice or professional trading performance.

Sequence used below: **Problem → formalization → assumptions → computation/estimation → validation → interpretation → limitations**.

Companion notes: `MODEL_RISK_NOTES.md` and `docs/data_policy.md`.

## 1. Coupon bond

### Problem

Value a default-free coupon bond and describe how the price moves when the constant yield used to discount it moves.

### Formalization

Cash flows are a coupon `C` at each payment date and face `F` at maturity. With annual payments and a flat yield `y`,

$$P(y) = \sum_{t=1}^{N} \frac{C}{(1+y)^t} + \frac{F}{(1+y)^N}.$$

Macaulay duration is the present-value-weighted average receipt time. Modified duration and convexity are the first and second derivatives of this map, rescaled so that

$$\frac{\Delta P}{P} \approx -D_{\mathrm{mod}}\,\Delta y + \tfrac{1}{2} C_{\mathrm{conv}} (\Delta y)^2.$$

### Assumptions

- Cash flows are fixed (no default, no optionality).
- One yield discounts every date (flat curve, parallel shocks).
- The local expansion is not a global description of large or non-parallel moves.

In [ ]:
import numpy as np

from qfinmodels.fixed_income import (
    bond_convexity,
    bond_price,
    duration_convexity_price,
    macaulay_duration,
    modified_duration,
)
from qfinmodels.plots import plot_duration_convexity
from qfinmodels.risk import expected_shortfall, historical_var, parametric_var
from qfinmodels.tvm import annuity_present_value
from qfinmodels.volatility import simulate_garch11

face, coupon_rate, years, ytm = 100.0, 0.05, 10.0, 0.06
price = bond_price(face, coupon_rate, years, ytm)
mac = macaulay_duration(face, coupon_rate, years, ytm)
mod = modified_duration(face, coupon_rate, years, ytm)
conv = bond_convexity(face, coupon_rate, years, ytm)
print(f"price = {price:.6f}")
print(f"Macaulay duration = {mac:.6f} years")
print(f"modified duration = {mod:.6f}")
print(f"convexity = {conv:.6f}")

annuity_check = annuity_present_value(5.0, 0.06, 10) + 100.0 * (1.06 ** -10)
print(f"annuity-plus-principal identity: {annuity_check:.6f}")

### Validation

The coupon-plus-principal sum must match the bond pricer. The duration-convexity formula is checked against a full reprice for a small shock and a large shock. The remainder should be smaller when the yield move is small.

In [ ]:
assert abs(price - annuity_check) < 1e-10

shocks = np.linspace(-0.02, 0.02, 21)
full = np.array([bond_price(face, coupon_rate, years, ytm + dy) for dy in shocks])
approx = np.array([duration_convexity_price(price, mod, conv, dy) for dy in shocks])

small, large = 0.001, 0.02
err_small = abs(duration_convexity_price(price, mod, conv, small) - bond_price(face, coupon_rate, years, ytm + small))
err_large = abs(duration_convexity_price(price, mod, conv, large) - bond_price(face, coupon_rate, years, ytm + large))
print(f"absolute error at +10bp:  {err_small:.8f}")
print(f"absolute error at +200bp: {err_large:.8f}")
assert err_small < err_large

fig = plot_duration_convexity(shocks, full, approx)
fig

### Interpretation and limitations

For this schedule, a higher yield lowers the price because every cash flow is a fixed claim. Duration describes a *local* parallel move. Convexity reduces the approximation error but still cannot speak to default, liquidity, or a twist in the curve. Those omissions are model risk, not rounding error.

## 2. Value-at-risk and expected shortfall

### Problem

Summarise one-period loss on a simulated return series at a stated probability level.

### Formalization

Let `R` be a simple return. Historical VaR at level `α` is `-q_α(R)`. Expected shortfall is `-E[R | R ≤ q_α(R)]`. Parametric VaR replaces the empirical law with `N(μ, σ²)` fitted on the same sample.

### Assumptions

- The sample is treated as the relevant loss distribution (historical) or as Gaussian (parametric).
- The horizon equals one observation spacing.
- The series below is a GARCH(1,1) draw, not a market history.

In [ ]:
returns, _ = simulate_garch11(1500, omega=0.00002, alpha=0.08, beta=0.88, seed=19)
alpha = 0.05
hvar = historical_var(returns, alpha=alpha)
pvar = parametric_var(returns, alpha=alpha)
es = expected_shortfall(returns, alpha=alpha)
print(f"historical VaR ({alpha:.0%}): {hvar:.6f}")
print(f"parametric VaR ({alpha:.0%}): {pvar:.6f}")
print(f"expected shortfall ({alpha:.0%}): {es:.6f}")
assert es >= hvar

### Validation, interpretation, limitations

On the same sample and level, expected shortfall is at least as large as historical VaR because ES averages the tail beyond the quantile. That inequality is a property of the definitions. It does not make either number a sufficient risk summary.

VaR is incomplete because:

- it is silent about loss given exceedance;
- it can fail subadditivity, so it is not a coherent risk measure;
- the level, window, and i.i.d. assumption are extra choices;
- parametric VaR inherits a Gaussian tail that the GARCH simulator itself violates once volatility clusters.

A green numerical check on simulated data is software validation. It is not evidence about a trading book. See `MODEL_RISK_NOTES.md`.